# LLFSR-Net v6 — Face Super-Resolution Under Low-Light
### LightEnhancerUNet (CBAM Attention) + GFPGAN v1.4 + Edge/SSIM Loss
**Key upgrades vs v5:**
- GFPGAN v1.4 (sharper, less grainy than v1.3)
- fidelity_weight 0.50 → 0.68 (eliminates speckle artifacts)
- Channel Attention in UNet bottleneck
- SSIM + Edge/Frequency loss for pixel-accurate crispness
- Mild sharpening + histogram correction post-GFPGAN
- 2500 train / 300 val samples · 20 epochs · ~18 min on T4

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q --upgrade pip
!pip install -q --upgrade basicsr torchvision
!pip install -q gfpgan facexlib realesrgan lpips scikit-image tqdm
print('All packages installed.')

## Cell 2 — Compatibility Patch + GPU Check

In [ ]:
# torchvision compatibility patch — must run BEFORE any basicsr import
import sys, types, torch, torchvision
from torchvision.transforms.functional import rgb_to_grayscale
fake = types.ModuleType('torchvision.transforms.functional_tensor')
fake.rgb_to_grayscale = rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = fake

if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'PyTorch  : {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print('Patch OK')

## Cell 3 — Imports + Seed

In [ ]:
import os, time, random, glob, shutil, warnings, zipfile, json as json_lib
warnings.filterwarnings('ignore')
import numpy as np, cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as tv_models
import lpips

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda')
print(f'Device: {DEVICE}  |  CUDA {torch.version.cuda}')

## Cell 4 — Config  ← SET YOUR CELEBA_PATH

In [ ]:
class Cfg:

    # ──────── SET THIS TO YOUR CELEBA ZIP PATH ────────
    CELEBA_PATH = '/content/drive/MyDrive/img_align_celeba.zip'

    OUT_DIR     = '/content/llfsr_v6'
    CACHE_DIR   = '/content/llfsr_v6/cache'
    DRIVE_CKPT  = '/content/drive/MyDrive/llfsr_v6_ckpts'
    WEIGHTS_DIR = '/content/llfsr_v6/weights'

    # Dataset — more samples for better generalisation
    N_TRAIN = 2500
    N_VAL   = 300
    HR_SIZE = 512   # Ground-truth resolution
    LR_SIZE = 128   # 4× downscale

    # Calibrated low-light degradation (realistic but not extreme)
    GAMMA_RANGE  = (0.28, 0.52)   # moderate darkening
    NOISE_SIGMA  = (0.002, 0.015) # very mild noise
    BLUR_PROB    = 0.30           # probability of adding subtle camera blur
    BLUR_KERNEL  = (3, 3)         # small kernel — simulates slight defocus

    # LightEnhancerUNet training
    ENH_LR     = 3e-4
    ENH_EPOCHS = 20      # +5 vs v5 — same wall-time due to efficiency gains
    ENH_BATCH  = 10      # slightly smaller to fit attention blocks in VRAM

    # GFPGAN fidelity — CRITICAL for sharp, non-grainy output
    # 0 = max GAN hallucination (grainy)  |  1 = max input fidelity (blurry)
    GFPGAN_FIDELITY = 0.68   # v5 was 0.50 which caused speckle artifacts

cfg = Cfg()
for d in [cfg.OUT_DIR, cfg.CACHE_DIR, cfg.WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Config ready.')
print(f'  LR input : {cfg.LR_SIZE}×{cfg.LR_SIZE}')
print(f'  HR output: {cfg.HR_SIZE}×{cfg.HR_SIZE}  ({cfg.HR_SIZE//cfg.LR_SIZE}× SR)')
print(f'  Train: {cfg.N_TRAIN}  Val: {cfg.N_VAL}  Epochs: {cfg.ENH_EPOCHS}')
print(f'  GFPGAN fidelity weight: {cfg.GFPGAN_FIDELITY}')

## Cell 5 — Mount Drive + Download GFPGAN v1.4 Weights

In [ ]:
import subprocess
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
os.makedirs(cfg.DRIVE_CKPT, exist_ok=True)

# ── GFPGAN v1.4 (better than v1.3: sharper details, less grain) ──
GFPGAN_PATH  = os.path.join(cfg.WEIGHTS_DIR, 'GFPGANv1.4.pth')
DRIVE_GFPGAN = '/content/drive/MyDrive/GFPGANv1.4.pth'

if os.path.exists(GFPGAN_PATH):
    print('GFPGAN v1.4 weights already cached locally.')
elif os.path.exists(DRIVE_GFPGAN):
    shutil.copy2(DRIVE_GFPGAN, GFPGAN_PATH)
    print('Copied GFPGAN v1.4 from Google Drive.')
else:
    print('Downloading GFPGAN v1.4 weights (~350 MB)...')
    url = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
    subprocess.run(f'wget -q --show-progress -O "{GFPGAN_PATH}" "{url}"',
                   shell=True, check=True)
    os.makedirs(os.path.dirname(DRIVE_GFPGAN), exist_ok=True)
    shutil.copy2(GFPGAN_PATH, DRIVE_GFPGAN)
    print('Saved to Drive for reuse.')

mb = os.path.getsize(GFPGAN_PATH) / 1e6
print(f'GFPGAN v1.4 ready  ({mb:.0f} MB)  →  {GFPGAN_PATH}')

## Cell 5b — Unzip CelebA (skipped if already done)

In [ ]:
celeba_dir = os.path.join(cfg.CACHE_DIR, 'img_align_celeba')

if not os.path.exists(celeba_dir) or not os.listdir(celeba_dir):
    print(f'Extracting CelebA to {celeba_dir}...')
    with zipfile.ZipFile(cfg.CELEBA_PATH, 'r') as z:
        members = [m for m in z.namelist()
                   if m.startswith('img_align_celeba/') and m.endswith('.jpg')]
        for m in tqdm(members, desc='Extracting', ncols=70):
            z.extract(m, cfg.CACHE_DIR)
    print(f'Extracted {len(members)} images.')
else:
    imgs_found = len(glob.glob(os.path.join(celeba_dir, '*.jpg')))
    print(f'Already extracted: {imgs_found} images in {celeba_dir}')

cfg.CELEBA_PATH = celeba_dir

## Cell 6 — Load Images to Local SSD + RAM

In [ ]:
all_paths = sorted(
    glob.glob(os.path.join(cfg.CELEBA_PATH, '*.jpg')) +
    glob.glob(os.path.join(cfg.CELEBA_PATH, '*.png'))
)
if not all_paths:
    raise FileNotFoundError(f'No images in {cfg.CELEBA_PATH}')
print(f'Found {len(all_paths)} CelebA images.')

random.shuffle(all_paths)
selected = all_paths[:cfg.N_TRAIN + cfg.N_VAL]

local_dir = os.path.join(cfg.CACHE_DIR, 'local')
os.makedirs(local_dir, exist_ok=True)
print(f'Copying {len(selected)} images to local SSD...')
local_paths = []
for p in tqdm(selected, desc='Copy', ncols=70):
    dst = os.path.join(local_dir, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy2(p, dst)
    local_paths.append(dst)


def load_images(paths, size, desc='Loading'):
    imgs = []
    for p in tqdm(paths, desc=f'{desc} {size}px', ncols=70):
        img = Image.open(p).convert('RGB')
        w, h = img.size; s = min(w, h)
        img  = img.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2))
        imgs.append(np.array(img.resize((size, size), Image.BICUBIC), dtype=np.uint8))
    return imgs


print('Loading 512px HR images into RAM...')
t0 = time.time()
hr_train = load_images(local_paths[:cfg.N_TRAIN], cfg.HR_SIZE, 'Train HR')
hr_val   = load_images(local_paths[cfg.N_TRAIN:], cfg.HR_SIZE, 'Val HR')
mb = sum(x.nbytes for x in hr_train + hr_val) / 1e6
print(f'Loaded {len(hr_train)+len(hr_val)} images in {time.time()-t0:.1f}s  ({mb:.0f} MB)')

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, ax in enumerate(axes):
    ax.imshow(hr_train[i * 400]); ax.axis('off')
    ax.set_title(f'Sample {i+1}', fontsize=10)
plt.suptitle('CelebA HR Training Samples (512×512)', fontsize=12)
plt.tight_layout(); plt.show()

## Cell 7 — Improved Degradation Pipeline
> New in v6: optional mild Gaussian blur to simulate camera defocus under low-light

In [ ]:
class LowLightDegradation:
    """
    Realistic low-light degradation for face SR training.
    Steps: gamma darkening -> colour cast -> optional blur -> Gaussian noise -> INTER_AREA downscale.
    """
    def __init__(self, cfg):
        self.cfg = cfg

    def __call__(self, hr_u8):
        cfg = self.cfg
        hr  = hr_u8.astype(np.float32) / 255.0

        # 1. Gamma darkening (simulates low exposure)
        g    = random.uniform(*cfg.GAMMA_RANGE)
        dark = np.power(hr, 1.0 / g)

        # 2. Subtle per-channel colour cast (realistic white-balance shift)
        cast = np.array([
            random.uniform(0.90, 1.10),
            random.uniform(0.93, 1.07),
            random.uniform(0.88, 1.12),
        ], dtype=np.float32)
        dark = np.clip(dark * cast, 0, 1)

        # 3. NEW: Optional mild Gaussian blur (camera defocus/shake)
        if random.random() < cfg.BLUR_PROB:
            dark_u8 = (dark * 255).astype(np.uint8)
            dark_u8 = cv2.GaussianBlur(dark_u8, cfg.BLUR_KERNEL, 0)
            dark = dark_u8.astype(np.float32) / 255.0

        # 4. Mild Gaussian noise
        sig  = random.uniform(*cfg.NOISE_SIGMA)
        dark = np.clip(dark + np.random.randn(*dark.shape).astype(np.float32) * sig, 0, 1)

        # 5. Downscale to LR with INTER_AREA (gold standard for downscaling)
        lr_u8       = cv2.resize((dark*255).astype(np.uint8),
                                  (cfg.LR_SIZE, cfg.LR_SIZE),
                                  interpolation=cv2.INTER_AREA)
        clean_lr_u8 = cv2.resize(hr_u8, (cfg.LR_SIZE, cfg.LR_SIZE),
                                  interpolation=cv2.INTER_AREA)

        def to_t(u8):
            return torch.from_numpy(u8).permute(2, 0, 1).float() / 255.0

        return to_t(lr_u8), to_t(clean_lr_u8), to_t(hr_u8)


class FaceDataset(Dataset):
    def __init__(self, imgs, cfg, augment=True):
        self.imgs = imgs
        self.deg  = LowLightDegradation(cfg)
        self.aug  = augment

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        if self.aug:
            # Horizontal flip
            if random.random() > 0.5:
                img = img[:, ::-1].copy()
            # Brightness jitter
            if random.random() > 0.6:
                img = np.clip(img.astype(np.float32) * random.uniform(0.88, 1.12),
                              0, 255).astype(np.uint8)
            # Saturation jitter
            if random.random() > 0.7:
                hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
                hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.85, 1.15), 0, 255)
                img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        return self.deg(img)


train_ds = FaceDataset(hr_train, cfg, augment=True)
val_ds   = FaceDataset(hr_val,   cfg, augment=False)

train_loader = DataLoader(train_ds, batch_size=cfg.ENH_BATCH, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True,
                           persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f'DataLoaders ready.  Train: {len(train_loader)} steps/epoch')

# Visualise degradation samples
N_VIS = 5
fig, axes = plt.subplots(2, N_VIS, figsize=(N_VIS*4.5, 9))
row_labels = ['Dark LR Input (128px)', 'Clean HR Ground Truth (512px)']
for col in range(N_VIS):
    dl, cl, hr = train_ds[col * 100]
    axes[0, col].imshow(np.clip(dl.permute(1,2,0).numpy(), 0, 1))
    axes[1, col].imshow(np.clip(hr.permute(1,2,0).numpy(), 0, 1))
    axes[0, col].set_title(f'Sample {col+1}', fontsize=10)
    for r in range(2): axes[r, col].axis('off')
for r, lbl in enumerate(row_labels):
    axes[r, 0].set_ylabel(lbl, fontsize=10, labelpad=6)
plt.suptitle('v6 Degradation: Dark 128px LR  →  Clean 512px HR', fontsize=13)
plt.tight_layout()
plt.savefig(f'{cfg.OUT_DIR}/degradation_v6.png', dpi=100, bbox_inches='tight')
plt.show()
print('Degradation preview saved.')

## Cell 8 — LightEnhancerUNet v2 with Channel Attention
> New in v6: CBAM-style channel attention in bottleneck helps the model focus on facial regions

In [ ]:
class DoubleConv(nn.Module):
    """Two Conv-BN-LeakyReLU blocks — standard UNet building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )
    def forward(self, x): return self.net(x)


class ChannelAttention(nn.Module):
    """
    CBAM channel attention — squeezes spatial dims, learns which feature
    channels matter most for low-light face enhancement.
    Adds ~0.03 M params at 256 channels.
    """
    def __init__(self, ch, ratio=8):
        super().__init__()
        mid = max(ch // ratio, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        scale = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        return x * scale


class SpatialAttention(nn.Module):
    """CBAM spatial attention — highlights face-region pixels."""
    def __init__(self, kernel=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel, padding=kernel//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = self.sigmoid(self.conv(torch.cat([avg_out, max_out], 1)))
        return x * scale


class CBAM(nn.Module):
    """Full CBAM block: channel attention then spatial attention."""
    def __init__(self, ch):
        super().__init__()
        self.ca = ChannelAttention(ch)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


class LightEnhancerUNet(nn.Module):
    """
    Improved UNet for low-light face enhancement at 128×128.
    Architecture: encoder (3→32→64→128→256) + CBAM bottleneck + skip-connected decoder.
    CBAM attention on bottleneck AND deepest decoder stage.
    ~1.7 M parameters — fits T4 at batch=10 with full loss suite.
    """
    def __init__(self, ch=(32, 64, 128, 256), scale=0.65):
        super().__init__()
        c = ch; self.scale = scale
        # Encoder
        self.enc1 = DoubleConv(3,    c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        # Bottleneck with CBAM
        self.bottleneck = DoubleConv(c[3], c[3])
        self.cbam_bn    = CBAM(c[3])           # <-- NEW: attention in bottleneck
        # Decoder (skip connections from each encoder stage)
        self.dec4 = DoubleConv(c[3]*2, c[2])
        self.cbam4 = CBAM(c[2])                # <-- NEW: attention in deepest decoder
        self.dec3 = DoubleConv(c[2]*2, c[1])
        self.dec2 = DoubleConv(c[1]*2, c[0])
        self.dec1 = DoubleConv(c[0]*2, c[0])
        # Ops
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        # Head — residual in [-1, 1]
        self.head = nn.Sequential(
            nn.Conv2d(c[0], 16, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(16, 3, 1),
            nn.Tanh()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.cbam_bn(self.bottleneck(self.pool(e4)))  # attention applied
        d4 = self.cbam4(self.dec4(torch.cat([self.up(b),  e4], 1)))
        d3 = self.dec3(torch.cat([self.up(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1))
        return (x + self.head(d1) * self.scale).clamp(0, 1)


enhancer = LightEnhancerUNet().to(DEVICE)
n_params  = sum(p.numel() for p in enhancer.parameters())
print(f'LightEnhancerUNet v2 (CBAM): {n_params/1e6:.2f} M parameters')

with torch.no_grad():
    dummy = torch.rand(2, 3, cfg.LR_SIZE, cfg.LR_SIZE).to(DEVICE)
    out   = enhancer(dummy)
    print(f'Forward test: {tuple(dummy.shape)} -> {tuple(out.shape)}')
print('LightEnhancerUNet v2 ready.')

## Cell 9 — Loss Functions: L1 + SSIM + VGG Perceptual + Edge
> New in v6: SSIM loss (structural fidelity) + Edge/Sobel loss (sharpness) — these are what give pixel-perfect crispness

In [ ]:
# ── VGG Perceptual Loss ────────────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    """
    Multi-scale perceptual loss using VGG-16 ImageNet features.
    relu1_2 -> local texture | relu2_2 -> edges | relu3_3 -> high-level structure.
    """
    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1)
        feats = list(vgg.features)
        self.s1 = nn.Sequential(*feats[:4]).eval()    # relu1_2
        self.s2 = nn.Sequential(*feats[4:9]).eval()   # relu2_2
        self.s3 = nn.Sequential(*feats[9:16]).eval()  # relu3_3
        for p in self.parameters(): p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def forward(self, pred, tgt):
        pred = (pred - self.mean) / self.std
        tgt  = (tgt  - self.mean) / self.std
        f1p = self.s1(pred);  f1t = self.s1(tgt)
        f2p = self.s2(f1p);   f2t = self.s2(f1t)
        f3p = self.s3(f2p);   f3t = self.s3(f2t)
        return F.l1_loss(f1p,f1t) + F.l1_loss(f2p,f2t) + F.l1_loss(f3p,f3t)


# ── NEW: SSIM Loss ──────────────────────────────────────────────────────────
class SSIMLoss(nn.Module):
    """
    Differentiable SSIM loss — penalises structural mismatches that L1 misses.
    Particularly effective for brightness/contrast restoration.
    """
    def __init__(self, window_size=11, C1=0.01**2, C2=0.03**2):
        super().__init__()
        self.ws = window_size
        self.C1 = C1; self.C2 = C2
        self.pad = window_size // 2
        # Gaussian window
        sigma = 1.5
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g /= g.sum()
        window = g.unsqueeze(1) * g.unsqueeze(0)  # [ws, ws]
        window = window.view(1, 1, window_size, window_size).repeat(3, 1, 1, 1)
        self.register_buffer('window', window)

    def forward(self, pred, tgt):
        def _ssim(x, y):
            mu_x  = F.conv2d(x, self.window, padding=self.pad, groups=3)
            mu_y  = F.conv2d(y, self.window, padding=self.pad, groups=3)
            mu_x2 = mu_x ** 2; mu_y2 = mu_y ** 2; mu_xy = mu_x * mu_y
            sx2 = F.conv2d(x*x, self.window, padding=self.pad, groups=3) - mu_x2
            sy2 = F.conv2d(y*y, self.window, padding=self.pad, groups=3) - mu_y2
            sxy = F.conv2d(x*y, self.window, padding=self.pad, groups=3) - mu_xy
            num = (2*mu_xy + self.C1) * (2*sxy + self.C2)
            den = (mu_x2 + mu_y2 + self.C1) * (sx2 + sy2 + self.C2)
            return torch.clamp(num / (den + 1e-8), -1, 1).mean()
        return 1.0 - _ssim(pred, tgt)


# ── NEW: Edge / Frequency Loss ──────────────────────────────────────────────
class EdgeFrequencyLoss(nn.Module):
    """
    Sobel-based edge loss — ensures predicted edges match GT edges.
    This is what gives crisp, pixel-accurate face details.
    Also includes a Laplacian for second-order sharpness.
    """
    def __init__(self):
        super().__init__()
        # Sobel X kernel
        sx = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]])
        sy = sx.t()
        # Laplacian kernel
        lap = torch.tensor([[0.,-1.,0.],[-1.,4.,-1.],[0.,-1.,0.]])
        # Expand to (out_ch, in_ch/groups, H, W) for depthwise conv
        self.register_buffer('sx',  sx.view(1,1,3,3).repeat(3,1,1,1))
        self.register_buffer('sy',  sy.view(1,1,3,3).repeat(3,1,1,1))
        self.register_buffer('lap', lap.view(1,1,3,3).repeat(3,1,1,1))

    def forward(self, pred, tgt):
        kw = dict(padding=1, groups=3)
        # Sobel magnitude
        ex_p = F.conv2d(pred, self.sx, **kw); ey_p = F.conv2d(pred, self.sy, **kw)
        ex_t = F.conv2d(tgt,  self.sx, **kw); ey_t = F.conv2d(tgt,  self.sy, **kw)
        em_p = torch.sqrt(ex_p**2 + ey_p**2 + 1e-6)
        em_t = torch.sqrt(ex_t**2 + ey_t**2 + 1e-6)
        sobel_loss = F.l1_loss(em_p, em_t)
        # Laplacian sharpness
        lp_p = F.conv2d(pred, self.lap, **kw)
        lp_t = F.conv2d(tgt,  self.lap, **kw)
        lap_loss = F.l1_loss(lp_p, lp_t)
        return sobel_loss + 0.5 * lap_loss


vgg_loss  = VGGPerceptualLoss().to(DEVICE)
ssim_loss = SSIMLoss().to(DEVICE)
edge_loss = EdgeFrequencyLoss().to(DEVICE)
print('All loss functions ready.')

# Loss weights — tuned for pixel-accurate, sharp, natural-looking output
W_L1   = 1.00   # pixel-accurate brightness & colour
W_SSIM = 0.50   # structural fidelity (contrast, luminance patterns)
W_VGG  = 0.08   # natural textures (prevents washout)
W_EDGE = 0.20   # sharpness: edges and fine detail  <-- NEW

print(f'\nLoss = L1×{W_L1} + SSIM×{W_SSIM} + VGG×{W_VGG} + Edge×{W_EDGE}')
print('  L1   -> pixel-accurate brightness/colour')
print('  SSIM -> structural fidelity and contrast')
print('  VGG  -> photorealistic natural textures')
print('  Edge -> crisp edges and fine facial detail  [NEW]')

## Cell 10 — Train LightEnhancerUNet  (~15–18 min on T4)

In [ ]:
opt    = torch.optim.AdamW(enhancer.parameters(), lr=cfg.ENH_LR,
                             betas=(0.9, 0.999), weight_decay=1e-4)
scaler = GradScaler()
BEST_PSNR = 0.0

# Resume from Drive checkpoint if available
ckpts = sorted(glob.glob(os.path.join(cfg.DRIVE_CKPT, 'enhancer_ep*.pth')))
start_ep = 0
if ckpts:
    ck = torch.load(ckpts[-1], map_location=DEVICE, weights_only=False)
    enhancer.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    start_ep = ck['epoch']; BEST_PSNR = ck.get('best_psnr', 0)
    print(f'Resumed from epoch {start_ep}  (PSNR {BEST_PSNR:.2f} dB)')
else:
    print('Training LightEnhancerUNet v2 from scratch...')

if start_ep < cfg.ENH_EPOCHS:
    total_steps = (cfg.ENH_EPOCHS - start_ep) * len(train_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.ENH_LR * 3, total_steps=total_steps,
        pct_start=0.10, anneal_strategy='cos',
        div_factor=10.0, final_div_factor=100.0
    )

    history = {'loss': [], 'psnr': [], 'ssim': []}
    print(f'Training epochs {start_ep+1} -> {cfg.ENH_EPOCHS}  |  '
          f'{len(train_loader)} steps/ep  |  batch={cfg.ENH_BATCH}')
    print('=' * 65)

    t_start = time.time()

    for epoch in range(start_ep + 1, cfg.ENH_EPOCHS + 1):
        enhancer.train(); ep_loss = 0.0; t0 = time.time()

        for dark_lr, clean_lr, _ in train_loader:
            dark_lr  = dark_lr.to(DEVICE,  non_blocking=True)
            clean_lr = clean_lr.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with autocast():
                enhanced = enhancer(dark_lr)
                loss = (W_L1   * F.l1_loss(enhanced, clean_lr)
                      + W_SSIM * ssim_loss(enhanced, clean_lr)
                      + W_VGG  * vgg_loss(enhanced, clean_lr)
                      + W_EDGE * edge_loss(enhanced, clean_lr))

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(enhancer.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            ep_loss += loss.item()

        ep_loss /= len(train_loader)
        history['loss'].append(ep_loss)

        # Validation PSNR / SSIM on enhanced LR (vs clean LR)
        enhancer.eval(); ps, ss = [], []
        with torch.no_grad():
            for dark_lr_v, clean_lr_v, _ in val_loader:
                dark_lr_v  = dark_lr_v.to(DEVICE)
                enhanced_v = enhancer(dark_lr_v)
                for j in range(enhanced_v.size(0)):
                    ep_np = enhanced_v[j].permute(1,2,0).cpu().numpy()
                    gt_np = clean_lr_v[j].permute(1,2,0).numpy()
                    ps.append(psnr_fn(gt_np, ep_np, data_range=1.0))
                    ss.append(ssim_fn(gt_np, ep_np, data_range=1.0, channel_axis=2))

        pv, sv = np.mean(ps), np.mean(ss)
        history['psnr'].append(pv); history['ssim'].append(sv)
        t_ep = (time.time() - t0) / 60
        print(f'Ep {epoch:02d}/{cfg.ENH_EPOCHS}  loss={ep_loss:.4f}  '
              f'PSNR={pv:.2f}dB  SSIM={sv:.4f}  {t_ep:.1f}min')

        # Save checkpoint
        ck_name  = f'enhancer_ep{epoch:03d}_psnr{pv:.2f}.pth'
        ck_local = os.path.join(cfg.OUT_DIR, ck_name)
        ck_drive = os.path.join(cfg.DRIVE_CKPT, ck_name)
        pkg = {'model': enhancer.state_dict(), 'opt': opt.state_dict(),
               'epoch': epoch, 'best_psnr': max(BEST_PSNR, pv)}
        torch.save(pkg, ck_local); shutil.copy2(ck_local, ck_drive)

        if pv > BEST_PSNR:
            BEST_PSNR = pv
            best_local = os.path.join(cfg.OUT_DIR, 'enhancer_best.pth')
            torch.save(pkg, best_local)
            shutil.copy2(best_local, os.path.join(cfg.DRIVE_CKPT, 'enhancer_best.pth'))
            print(f'  ★ NEW BEST  PSNR = {BEST_PSNR:.2f} dB')

    total_min = (time.time() - t_start) / 60
    print(f'\nTraining complete  ({total_min:.1f} min)  |  Best PSNR: {BEST_PSNR:.2f} dB')

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(history['loss'],  color='steelblue');    ax[0].set_title('Train Loss');    ax[0].grid(alpha=0.3)
    ax[1].plot(history['psnr'],  'o-', color='green'); ax[1].set_title('Val PSNR (dB)'); ax[1].grid(alpha=0.3)
    ax[2].plot(history['ssim'],  's-', color='purple');ax[2].set_title('Val SSIM');      ax[2].grid(alpha=0.3)
    plt.suptitle('LightEnhancerUNet v2 Training Curves', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{cfg.OUT_DIR}/enhancer_curves_v6.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print(f'Training already completed for {cfg.ENH_EPOCHS} epochs.')

## Cell 11 — Load Best Weights + GFPGAN v1.4 + Post-Processing Pipeline
> New in v6: histogram matching + mild unsharp mask after GFPGAN — eliminates colour drift and restores crisp detail

In [ ]:
# ── Load best LightEnhancerUNet checkpoint ────────────────────────────────
best_ck = os.path.join(cfg.OUT_DIR, 'enhancer_best.pth')
if not os.path.exists(best_ck):
    drive_best = os.path.join(cfg.DRIVE_CKPT, 'enhancer_best.pth')
    if os.path.exists(drive_best):
        shutil.copy2(drive_best, best_ck)
    else:
        raise FileNotFoundError('Best checkpoint not found. Run Cell 10 first.')
ck = torch.load(best_ck, map_location=DEVICE, weights_only=False)
enhancer.load_state_dict(ck['model'])
enhancer.eval()
print(f'LightEnhancerUNet loaded  (best val PSNR = {ck["best_psnr"]:.2f} dB)')

# ── Initialise GFPGAN v1.4 ────────────────────────────────────────────────
from gfpgan import GFPGANer

gfpganer = GFPGANer(
    model_path         = GFPGAN_PATH,
    upscale            = 4,      # 128 × 4 = 512
    arch               = 'clean',
    channel_multiplier = 2,
    bg_upsampler       = None    # skip background SR for speed
)
print(f'GFPGAN v1.4 restorer ready  ({cfg.LR_SIZE}→{cfg.HR_SIZE}px)')


# ── NEW: Post-processing helpers ──────────────────────────────────────────
def histogram_match_lab(src, ref):
    """
    Match the L*a*b* statistics of src to ref.
    src, ref : uint8 RGB numpy arrays of the same size.
    Corrects colour drift introduced by GFPGAN hallucination.
    """
    src_lab = cv2.cvtColor(src, cv2.COLOR_RGB2LAB).astype(np.float32)
    ref_lab = cv2.cvtColor(ref, cv2.COLOR_RGB2LAB).astype(np.float32)
    for c in range(3):
        s_mean, s_std = src_lab[:,:,c].mean(), src_lab[:,:,c].std() + 1e-6
        r_mean, r_std = ref_lab[:,:,c].mean(), ref_lab[:,:,c].std() + 1e-6
        src_lab[:,:,c] = (src_lab[:,:,c] - s_mean) * (r_std / s_std) + r_mean
    matched = np.clip(src_lab, 0, 255).astype(np.uint8)
    return cv2.cvtColor(matched, cv2.COLOR_LAB2RGB)


def unsharp_mask(img, sigma=1.2, strength=0.35):
    """
    Mild unsharp mask — restores high-frequency crispness.
    strength=0.35 is subtle; increase to 0.5 for more sharpening.
    """
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    sharpened = cv2.addWeighted(img, 1.0 + strength, blurred, -strength, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def run_gfpgan(enh_u8_rgb, weight=None):
    """
    Full enhanced pipeline:
      1. GFPGAN v1.4 face restoration + 4× upscale
      2. Histogram matching to enhanced input (fixes colour drift)
      3. Mild unsharp mask (recovers crisp detail)

    enh_u8_rgb : H×W×3 uint8 RGB (enhanced LR from LightEnhancerUNet)
    weight     : GFPGAN fidelity — overrides cfg.GFPGAN_FIDELITY if provided
    Returns    : 512×512×3 uint8 RGB
    """
    w = weight if weight is not None else cfg.GFPGAN_FIDELITY
    bgr = cv2.cvtColor(enh_u8_rgb, cv2.COLOR_RGB2BGR)
    _, _, restored_bgr = gfpganer.enhance(
        bgr, has_aligned=False, only_center_face=False,
        paste_back=True, weight=w
    )
    sr_rgb = cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB)

    # Upscale enhanced to 512 for colour reference
    enh_up = cv2.resize(enh_u8_rgb, (cfg.HR_SIZE, cfg.HR_SIZE), cv2.INTER_LANCZOS4)

    # 1. Colour correction via LAB histogram matching
    sr_rgb = histogram_match_lab(sr_rgb, enh_up)

    # 2. Mild unsharp mask for crisp edges
    sr_rgb = unsharp_mask(sr_rgb, sigma=1.2, strength=0.35)

    return sr_rgb


def full_pipeline_v6(dark_lr_batch, weight=None):
    """
    dark_lr_batch : [B, 3, 128, 128] float32 in [0,1]
    Returns       : list of dict with 'enhanced' (128px) and 'sr' (512px), both uint8 RGB
    """
    enhancer.eval()
    with torch.no_grad():
        enhanced_batch = enhancer(dark_lr_batch)
    out = []
    for i in range(enhanced_batch.size(0)):
        enh_np = (enhanced_batch[i].permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
        sr_np  = run_gfpgan(enh_np, weight=weight)
        out.append({'enhanced': enh_np, 'sr': sr_np})
    return out


# ── Sanity check ─────────────────────────────────────────────────────────
print('\nRunning sanity check on one validation image...')
dark_lr_s, clean_lr_s, clean_hr_s = val_ds[0]
result_s = full_pipeline_v6(dark_lr_s.unsqueeze(0).to(DEVICE))[0]
enh_s, sr_s = result_s['enhanced'], result_s['sr']
gt_s  = (clean_hr_s.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)
dl_s  = (dark_lr_s.permute(1,2,0).numpy()  * 255).clip(0,255).astype(np.uint8)
pv_s  = psnr_fn(gt_s/255., sr_s/255., data_range=1.0)
sv_s  = ssim_fn(gt_s/255., sr_s/255., data_range=1.0, channel_axis=2)
print(f'  Input: {dl_s.shape}  ->  Enhanced: {enh_s.shape}  ->  SR: {sr_s.shape}')
print(f'  PSNR = {pv_s:.2f} dB  |  SSIM = {sv_s:.4f}')

# 4-panel preview
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
dl_show  = cv2.resize(dl_s,  (512,512), cv2.INTER_NEAREST)
enh_show = cv2.resize(enh_s, (512,512), cv2.INTER_NEAREST)
bic_s    = cv2.resize(dl_s,  (512,512), cv2.INTER_CUBIC)
for ax, (im, tl, col) in zip(axes, [
    (dl_show,  'Dark LR Input\n(128px shown at 512)',     'black'),
    (enh_show, 'LightEnhancerUNet v2\n(128px shown at 512)', 'black'),
    (bic_s,    'Bicubic Baseline\n(128→512)',              'black'),
    (sr_s,     f'GFPGAN v1.4 SR [OURS]\nPSNR={pv_s:.1f}dB  SSIM={sv_s:.3f}', '#006400'),
]):
    ax.imshow(im); ax.axis('off')
    ax.set_title(tl, fontsize=9.5, color=col, fontweight='bold' if col != 'black' else 'normal')
for sp in axes[3].spines.values():
    sp.set_visible(True); sp.set_edgecolor('#00cc44'); sp.set_linewidth(4)
plt.suptitle('Sanity Check — Full Pipeline v6 (GFPGAN v1.4 + Post-Processing)', fontsize=13)
plt.tight_layout(); plt.show()

## Cell 12 — Full Validation Evaluation + Metrics

In [ ]:
print('Running full pipeline on all validation images...')
print(f'({len(val_ds)} images, GFPGAN ~0.5s/img, ~{len(val_ds)//120:.0f}-{len(val_ds)//90:.0f} min)')

all_psnr, all_ssim, all_lpips = [], [], []
vis_results = []   # (psnr, ssim, dl, enh, sr, gt, bic)
lpips_fn = lpips.LPIPS(net='alex').to(DEVICE).eval()

t0 = time.time()
with torch.no_grad():
    for dark_lr, clean_lr, clean_hr in tqdm(val_loader, desc='Eval', ncols=70):
        dark_lr_gpu  = dark_lr.to(DEVICE)
        enhanced_gpu = enhancer(dark_lr_gpu)

        for j in range(dark_lr.size(0)):
            enh_np = (enhanced_gpu[j].permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
            sr_np  = run_gfpgan(enh_np)
            gt_np  = (clean_hr[j].permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)
            dl_np  = (dark_lr[j].permute(1,2,0).numpy()  * 255).clip(0,255).astype(np.uint8)
            bic_np = cv2.resize(dl_np, (cfg.HR_SIZE, cfg.HR_SIZE), cv2.INTER_CUBIC)

            pv = psnr_fn(gt_np/255., sr_np/255., data_range=1.0)
            sv = ssim_fn(gt_np/255., sr_np/255., data_range=1.0, channel_axis=2)
            all_psnr.append(pv); all_ssim.append(sv)

            sr_t = torch.from_numpy(sr_np).permute(2,0,1).float().unsqueeze(0)/127.5-1
            gt_t = torch.from_numpy(gt_np).permute(2,0,1).float().unsqueeze(0)/127.5-1
            all_lpips.append(lpips_fn(sr_t.to(DEVICE), gt_t.to(DEVICE)).item())

            vis_results.append((pv, sv, dl_np, enh_np, sr_np, gt_np, bic_np))

elapsed = (time.time() - t0) / 60
print(f'\nEvaluation complete ({elapsed:.1f} min)')
print('=' * 58)
print(f'  n = {len(all_psnr)} validation images')
print(f'  PSNR  : {np.mean(all_psnr):.2f} +/- {np.std(all_psnr):.2f} dB')
print(f'  SSIM  : {np.mean(all_ssim):.4f} +/- {np.std(all_ssim):.4f}')
print(f'  LPIPS : {np.mean(all_lpips):.4f} +/- {np.std(all_lpips):.4f}  (lower is better)')
print('=' * 58)

# Metric histograms
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title, color in zip(axes,
    [all_psnr, all_ssim, all_lpips],
    ['PSNR (dB)', 'SSIM', 'LPIPS (lower=better)'],
    ['steelblue', 'seagreen', 'tomato']):
    ax.hist(data, bins=25, color=color, alpha=0.82, edgecolor='white')
    ax.axvline(np.mean(data), color='black', ls='--', lw=2,
               label=f'Mean = {np.mean(data):.3f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.suptitle('LLFSR-Net v6 — Metric Distributions (Val Set)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{cfg.OUT_DIR}/metrics_v6.png', dpi=110, bbox_inches='tight')
plt.show()

## Cell 13 — Qualitative Comparison Grid (9 rows × 5 cols)

In [ ]:
vis_results.sort(key=lambda x: x[0])   # ascending PSNR
best3  = vis_results[-3:][::-1]
worst3 = vis_results[:3]
mid    = len(vis_results) // 2
rand3  = random.sample(vis_results[max(0,mid-20):mid+20], 3)

groups = [
    ('Best Results',   best3,  '#1a7f37'),
    ('Random Samples', rand3,  '#0969da'),
    ('Hardest Cases',  worst3, '#cf222e'),
]

HDR = ['Dark LR Input\n(128px)',
       'LightEnhancer v2\n(128→512 display)',
       'Bicubic Baseline\n(128→512)',
       'GFPGAN v1.4 SR [OURS]\n(512px)',
       'Ground Truth\n(512px)']

fig = plt.figure(figsize=(30, 42))
gs  = gridspec.GridSpec(9, 5, hspace=0.20, wspace=0.03)

row = 0
for gi, (gname, items, gcolor) in enumerate(groups):
    for ri, (pv, sv, dl, en, sr, gt, bic) in enumerate(items):
        dl_show  = cv2.resize(dl, (512, 512), cv2.INTER_NEAREST)
        enh_show = cv2.resize(en, (512, 512), cv2.INTER_NEAREST)
        imgs     = [dl_show, enh_show, bic, sr, gt]

        for col, im in enumerate(imgs):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(im.astype(np.uint8)); ax.axis('off')
            if col == 3:
                for sp in ax.spines.values():
                    sp.set_visible(True); sp.set_edgecolor('#00cc44'); sp.set_linewidth(4)
            if ri == 0:
                c = '#006400' if col == 3 else 'black'
                ax.set_title(HDR[col], fontsize=9, fontweight='bold', color=c, pad=4)
            if col == 0:
                ax.set_ylabel(
                    f'{gname}\nPSNR={pv:.2f}dB\nSSIM={sv:.3f}',
                    fontsize=8, rotation=0, labelpad=115, va='center',
                    color=gcolor, fontweight='bold'
                )
        row += 1

plt.suptitle(
    'LLFSR-Net v6  ·  LightEnhancerUNet v2 (CBAM) + GFPGAN v1.4  ·  Dark 128px → Restored 512px',
    fontsize=14, y=1.002, fontweight='bold'
)
plt.savefig(f'{cfg.OUT_DIR}/comparison_grid_v6.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Comparison grid saved → {cfg.OUT_DIR}/comparison_grid_v6.png')

## Cell 14 — Test on Your Own Image (Simulated Low-Light)

In [ ]:
from google.colab import files as colab_files

print('Upload a face image to test the full v6 pipeline (will simulate low-light):')
uploaded = colab_files.upload()

for fname, data in (uploaded or {}).items():
    buf = np.frombuffer(data, np.uint8)
    img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]; s = min(h, w)
    hr   = cv2.resize(img[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2],
                       (cfg.HR_SIZE, cfg.HR_SIZE), cv2.INTER_CUBIC)

    # Simulate moderate low-light (gamma=0.40, mild noise)
    dark = (np.power(hr.astype(np.float32)/255.0, 0.40) * 255).clip(0,255).astype(np.uint8)
    dark = np.clip(dark.astype(np.float32) + np.random.randn(*dark.shape)*3, 0,255).astype(np.uint8)
    dark_lr = cv2.resize(dark, (cfg.LR_SIZE, cfg.LR_SIZE), cv2.INTER_AREA)

    t = torch.from_numpy(dark_lr).permute(2,0,1).float().unsqueeze(0) / 255.
    with torch.no_grad():
        enh = enhancer(t.to(DEVICE))
    enh_np = (enh[0].permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
    sr_np  = run_gfpgan(enh_np)
    bic_np = cv2.resize(dark_lr, (cfg.HR_SIZE, cfg.HR_SIZE), cv2.INTER_CUBIC)

    pv = psnr_fn(hr/255., sr_np/255., data_range=1.0)
    sv = ssim_fn(hr/255., sr_np/255., data_range=1.0, channel_axis=2)

    dl_show  = cv2.resize(dark_lr, (256,256), cv2.INTER_NEAREST)
    enh_show = cv2.resize(enh_np,  (256,256), cv2.INTER_NEAREST)

    fig, axes = plt.subplots(1, 5, figsize=(32, 6))
    titles = [
        'Dark LR Input\n(128px → 256 display)',
        'LightEnhancer v2\n(128px → 256 display)',
        'Bicubic Baseline\n(128 → 512px)',
        f'GFPGAN v1.4 [OURS]\n512px | PSNR={pv:.1f}dB SSIM={sv:.3f}',
        'Original HR\n(reference)',
    ]
    data_list = [dl_show, enh_show, bic_np, sr_np, hr]
    for ax, im, tl in zip(axes, data_list, titles):
        ax.imshow(im); ax.axis('off'); ax.set_title(tl, fontsize=9)
    axes[3].set_title(titles[3], fontsize=9, color='#006400', fontweight='bold')
    for sp in axes[3].spines.values():
        sp.set_visible(True); sp.set_edgecolor('#00cc44'); sp.set_linewidth(3)
    plt.suptitle(f'Custom Image Test (v6): {fname}', fontsize=12)
    plt.tight_layout()
    out_path = f'{cfg.OUT_DIR}/custom_{os.path.splitext(fname)[0]}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.show()
    Image.fromarray(sr_np).save(f'{cfg.OUT_DIR}/custom_{os.path.splitext(fname)[0]}_SR512.png')
    print(f'SR image saved: {cfg.OUT_DIR}/custom_{os.path.splitext(fname)[0]}_SR512.png')

## Cell 15 — Test on a Real Dark / Night Face Photo

In [ ]:
print('Upload a real dark face photo (no simulation applied):')
real = colab_files.upload()

for fname, data in (real or {}).items():
    buf  = np.frombuffer(data, np.uint8)
    img  = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]; s = min(h, w)
    crop = img[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]
    lr   = cv2.resize(crop, (cfg.LR_SIZE, cfg.LR_SIZE), cv2.INTER_AREA)

    t = torch.from_numpy(lr).permute(2,0,1).float().unsqueeze(0) / 255.
    with torch.no_grad():
        enh = enhancer(t.to(DEVICE))
    enh_np = (enh[0].permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
    sr_np  = run_gfpgan(enh_np)
    bic_np = cv2.resize(lr, (cfg.HR_SIZE, cfg.HR_SIZE), cv2.INTER_CUBIC)

    lr_show  = cv2.resize(lr,     (256,256), cv2.INTER_NEAREST)
    enh_show = cv2.resize(enh_np, (256,256), cv2.INTER_NEAREST)

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    for ax, (im, tl) in zip(axes, [
        (lr_show,  'Real Dark Input\n(128px → 256 display)'),
        (enh_show, 'LightEnhancer v2\n(128px → 256 display)'),
        (bic_np,   'Bicubic Baseline\n(128 → 512px)'),
        (sr_np,    'GFPGAN v1.4 SR [OURS]\n(512px)'),
    ]):
        ax.imshow(im); ax.axis('off'); ax.set_title(tl, fontsize=10)
    axes[3].set_title('GFPGAN v1.4 SR [OURS]\n(512px)', fontsize=10,
                       color='#006400', fontweight='bold')
    for sp in axes[3].spines.values():
        sp.set_visible(True); sp.set_edgecolor('#00cc44'); sp.set_linewidth(3)
    plt.suptitle(f'Real Low-Light Face Enhancement (v6): {fname}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{cfg.OUT_DIR}/real_{os.path.splitext(fname)[0]}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    Image.fromarray(sr_np).save(f'{cfg.OUT_DIR}/real_{os.path.splitext(fname)[0]}_SR512.png')
    print(f'SR saved: {cfg.OUT_DIR}/real_{os.path.splitext(fname)[0]}_SR512.png')

## Cell 16 — (Optional) Fidelity Weight Sweep
> Sweep GFPGAN fidelity_weight to visually find your preferred sharpness/naturalness tradeoff

In [ ]:
# Run a sweep on one image to visualise the fidelity tradeoff
dark_lr_sw, _, clean_hr_sw = val_ds[5]
with torch.no_grad():
    enh_sw = enhancer(dark_lr_sw.unsqueeze(0).to(DEVICE))
enh_sw_np = (enh_sw[0].permute(1,2,0).cpu().numpy() * 255).clip(0,255).astype(np.uint8)
gt_sw_np  = (clean_hr_sw.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8)

weights = [0.3, 0.5, 0.65, 0.8, 1.0]
results_sw = []
for w in weights:
    sr = run_gfpgan(enh_sw_np, weight=w)
    pv = psnr_fn(gt_sw_np/255., sr/255., data_range=1.0)
    sv = ssim_fn(gt_sw_np/255., sr/255., data_range=1.0, channel_axis=2)
    results_sw.append((sr, pv, sv))

fig, axes = plt.subplots(1, len(weights)+1, figsize=((len(weights)+1)*5.5, 5.5))
axes[0].imshow(gt_sw_np); axes[0].axis('off')
axes[0].set_title('Ground Truth\n(512px)', fontsize=10)
for ax, w, (sr, pv, sv) in zip(axes[1:], weights, results_sw):
    ax.imshow(sr); ax.axis('off')
    marker = ' ← cfg' if abs(w - cfg.GFPGAN_FIDELITY) < 0.01 else ''
    ax.set_title(f'fidelity={w}{marker}\nPSNR={pv:.1f}dB SSIM={sv:.3f}',
                 fontsize=9, color='#006400' if marker else 'black')
plt.suptitle('GFPGAN Fidelity Weight Sweep — Lower=more GAN, Higher=more input', fontsize=12)
plt.tight_layout()
plt.savefig(f'{cfg.OUT_DIR}/fidelity_sweep_v6.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Current cfg fidelity: {cfg.GFPGAN_FIDELITY}')
print('To change globally: cfg.GFPGAN_FIDELITY = <your chosen value>')

## Cell 17 — Save Models + Download for Deployment

In [ ]:
# ── Save LightEnhancerUNet ────────────────────────────────────────────────
inf_path = os.path.join(cfg.OUT_DIR, 'llfsr_enhancer_v6.pth')
torch.save({
    'model_state': enhancer.state_dict(),
    'model_class': 'LightEnhancerUNet_v2',
    'channels':    (32, 64, 128, 256),
    'scale':       0.65,
    'lr_size':     cfg.LR_SIZE,
    'hr_size':     cfg.HR_SIZE,
    'description': 'LightEnhancerUNet v2 (CBAM) — Stage 1 of LLFSR-Net v6',
}, inf_path)
shutil.copy2(inf_path, '/content/drive/MyDrive/llfsr_enhancer_v6.pth')
print(f'LightEnhancerUNet saved  ({os.path.getsize(inf_path)/1e6:.1f} MB)')

# ── Pipeline config ────────────────────────────────────────────────────────
pipe_cfg = {
    'version':            'v6',
    'lr_size':            cfg.LR_SIZE,
    'hr_size':            cfg.HR_SIZE,
    'pipeline':           'LightEnhancerUNet v2 (CBAM) -> GFPGAN v1.4 -> HistMatch + Sharpen',
    'gfpgan_version':     '1.4',
    'gfpgan_upscale':     4,
    'gfpgan_arch':        'clean',
    'gfpgan_fidelity_w':  cfg.GFPGAN_FIDELITY,
    'post_processing':    'LAB histogram matching + unsharp mask (sigma=1.2, strength=0.35)',
    'losses':             'L1 + SSIM + VGG + Edge',
    'n_train':            cfg.N_TRAIN,
    'epochs':             cfg.ENH_EPOCHS,
}
cfg_path = os.path.join(cfg.OUT_DIR, 'pipeline_config_v6.json')
with open(cfg_path, 'w') as f:
    json_lib.dump(pipe_cfg, f, indent=2)
shutil.copy2(cfg_path, '/content/drive/MyDrive/llfsr_pipeline_config_v6.json')
print('Pipeline config saved.')

# ── UI inference script ────────────────────────────────────────────────────
UI_CODE = '''#!/usr/bin/env python3
"""
LLFSR-Net v6 — UI Inference Script
LightEnhancerUNet v2 (CBAM) + GFPGAN v1.4 + Post-Processing

Install:
    pip install torch torchvision gfpgan basicsr facexlib opencv-python

Usage:
    from ui_inference_v6 import LLFSRPipelineV6
    pipeline = LLFSRPipelineV6(\'llfsr_enhancer_v6.pth\', \'GFPGANv1.4.pth\')
    result_bgr = pipeline.enhance(input_bgr_img)
"""
import torch, torch.nn as nn, torch.nn.functional as F
import cv2, numpy as np


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,1,1,bias=False),nn.BatchNorm2d(out_ch),nn.LeakyReLU(0.1,True),
            nn.Conv2d(out_ch,out_ch,3,1,1,bias=False),nn.BatchNorm2d(out_ch),nn.LeakyReLU(0.1,True),
        )
    def forward(self,x): return self.net(x)


class ChannelAttention(nn.Module):
    def __init__(self,ch,ratio=8):
        super().__init__()
        mid=max(ch//ratio,8)
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Conv2d(ch,mid,1,bias=False),nn.ReLU(True),nn.Conv2d(mid,ch,1,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x): return x*self.sig(self.fc(self.avg(x))+self.fc(self.max(x)))


class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([torch.mean(x,1,True),torch.max(x,1,True)[0]],1)))


class CBAM(nn.Module):
    def __init__(self,ch): super().__init__(); self.ca=ChannelAttention(ch); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))


class LightEnhancerUNet(nn.Module):
    def __init__(self,ch=(32,64,128,256),scale=0.65):
        super().__init__()
        c=ch; self.scale=scale
        self.enc1=DoubleConv(3,c[0]); self.enc2=DoubleConv(c[0],c[1])
        self.enc3=DoubleConv(c[1],c[2]); self.enc4=DoubleConv(c[2],c[3])
        self.bottleneck=DoubleConv(c[3],c[3]); self.cbam_bn=CBAM(c[3])
        self.dec4=DoubleConv(c[3]*2,c[2]); self.cbam4=CBAM(c[2])
        self.dec3=DoubleConv(c[2]*2,c[1]); self.dec2=DoubleConv(c[1]*2,c[0]); self.dec1=DoubleConv(c[0]*2,c[0])
        self.pool=nn.MaxPool2d(2)
        self.up=nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)
        self.head=nn.Sequential(nn.Conv2d(c[0],16,3,1,1),nn.LeakyReLU(0.1,True),nn.Conv2d(16,3,1),nn.Tanh())
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.cbam_bn(self.bottleneck(self.pool(e4)))
        d4=self.cbam4(self.dec4(torch.cat([self.up(b),e4],1)))
        d3=self.dec3(torch.cat([self.up(d4),e3],1))
        d2=self.dec2(torch.cat([self.up(d3),e2],1))
        d1=self.dec1(torch.cat([self.up(d2),e1],1))
        return (x+self.head(d1)*self.scale).clamp(0,1)


def histogram_match_lab(src, ref):
    src_lab=cv2.cvtColor(src,cv2.COLOR_RGB2LAB).astype(np.float32)
    ref_lab=cv2.cvtColor(ref,cv2.COLOR_RGB2LAB).astype(np.float32)
    for c in range(3):
        sm,ss=src_lab[:,:,c].mean(),src_lab[:,:,c].std()+1e-6
        rm,rs=ref_lab[:,:,c].mean(),ref_lab[:,:,c].std()+1e-6
        src_lab[:,:,c]=(src_lab[:,:,c]-sm)*(rs/ss)+rm
    return cv2.cvtColor(np.clip(src_lab,0,255).astype(np.uint8),cv2.COLOR_LAB2RGB)


def unsharp_mask(img,sigma=1.2,strength=0.35):
    b=cv2.GaussianBlur(img,(0,0),sigma)
    return np.clip(cv2.addWeighted(img,1+strength,b,-strength,0),0,255).astype(np.uint8)


class LLFSRPipelineV6:
    LR=128; HR=512

    def __init__(self,enhancer_ckpt,gfpgan_ckpt,fidelity_weight=0.68,device=None):
        self.w=fidelity_weight
        self.device=device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.enhancer=LightEnhancerUNet().to(self.device).eval()
        ck=torch.load(enhancer_ckpt,map_location=self.device,weights_only=False)
        self.enhancer.load_state_dict(ck["model_state"])
        from gfpgan import GFPGANer
        self.gfpganer=GFPGANer(model_path=gfpgan_ckpt,upscale=4,arch="clean",
                               channel_multiplier=2,bg_upsampler=None)
        print(f"LLFSRPipelineV6 ready on {self.device}")

    def enhance(self,img_bgr):
        h,w=img_bgr.shape[:2]; s=min(h,w)
        crop=img_bgr[(h-s)//2:(h+s)//2,(w-s)//2:(w+s)//2]
        lr=cv2.resize(crop,(self.LR,self.LR),cv2.INTER_AREA)
        lr_rgb=cv2.cvtColor(lr,cv2.COLOR_BGR2RGB)
        t=torch.from_numpy(lr_rgb).permute(2,0,1).float().unsqueeze(0)/255.
        with torch.no_grad(): enh=self.enhancer(t.to(self.device))
        enh_np=(enh[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
        enh_bgr=cv2.cvtColor(enh_np,cv2.COLOR_RGB2BGR)
        _,_,sr_bgr=self.gfpganer.enhance(enh_bgr,has_aligned=False,paste_back=True,weight=self.w)
        sr_rgb=cv2.cvtColor(sr_bgr,cv2.COLOR_BGR2RGB)
        enh_up=cv2.resize(enh_np,(self.HR,self.HR),cv2.INTER_LANCZOS4)
        sr_rgb=histogram_match_lab(sr_rgb,enh_up)
        sr_rgb=unsharp_mask(sr_rgb)
        return cv2.cvtColor(sr_rgb,cv2.COLOR_RGB2BGR)
'''

ui_path = os.path.join(cfg.OUT_DIR, 'ui_inference_v6.py')
with open(ui_path, 'w') as f:
    f.write(UI_CODE)
shutil.copy2(ui_path, '/content/drive/MyDrive/llfsr_ui_inference_v6.py')
print('UI inference script saved.')

# ── Download all files ─────────────────────────────────────────────────────
from google.colab import files as colab_files
print('\nDownloading model files...')
colab_files.download(inf_path)    # LightEnhancerUNet v2 weights (~7 MB)
colab_files.download(GFPGAN_PATH) # GFPGAN v1.4 weights (~350 MB)
colab_files.download(cfg_path)    # pipeline_config_v6.json
colab_files.download(ui_path)     # ui_inference_v6.py
print('\nAll files downloaded!')
print('\nQuick usage:')
print('  from ui_inference_v6 import LLFSRPipelineV6')
print('  pipe = LLFSRPipelineV6(\'llfsr_enhancer_v6.pth\', \'GFPGANv1.4.pth\')')
print('  result_bgr = pipe.enhance(cv2.imread(\'dark_face.jpg\'))')